In [ ]:
!pip install pymanopt

In [ ]:
import numpy as np
import tensorflow as tf
import cupy as cp
import jax.numpy as jnp
import jax
from jax import device_put
from jax.lib import xla_bridge

import pymanopt
from pymanopt.manifolds import Oblique
from pymanopt.optimizers import TrustRegions

import time
import gc

In [ ]:
def reset_memory():
    # Delete all possible GPU arrays
    globals_ = list(globals().keys())
    for var in globals_:
        if var in ['cp', 'jnp', 'reset_memory']: continue
        if isinstance(globals()[var], (cp.ndarray, jnp.ndarray)):
            del globals()[var]

    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()

In [ ]:
def manopt(A,B,maxtime):
    r,n = B.shape
    print("n",n,"r",r)
    manifold = Oblique(r,n)

    @pymanopt.function.jax(manifold)
    def cost(X):
        return -jnp.trace(X @ A @ X.T)


    @pymanopt.function.jax(manifold)
    def euclidean_gradient(X):
        return -2.0 * (X @ A)

    @pymanopt.function.jax(manifold)
    def euclidean_hessian(X, eta):
        return -2.0 * (eta @ A)

    # euclidean_gradient = euclidean_hessian = None

    problem = pymanopt.Problem(
        manifold,
        cost,
        euclidean_gradient=euclidean_gradient,
        euclidean_hessian=euclidean_hessian,
    )

    optimizer = TrustRegions(verbosity=1,max_time = maxtime)
    solution = optimizer.run(problem,initial_point= B)
    return problem,optimizer,solution

In [ ]:
n = 30000
k = int(np.ceil((n * 2) ** 0.5 ))

repeat = 50

f_rtr = np.zeros((repeat,3))
f_cg = np.zeros((repeat,3))

t_rtr = np.zeros((repeat,3))
t_cg = np.zeros((repeat,3))

shift = 50/n

# generate_B = cp.random.RandomState(seed = 0)
cp.random.seed(0)
for rep in range(repeat):
  print("----------------REP++",rep,"++---------------------------------------")
  reset_memory()
  C = cp.random.randn(n,n)/n
  C = C + C.T
  cp.fill_diagonal(C, C.diagonal() + shift)

  # generate_B.seed = 0
  # B = generate_B.randn(k,n)
  B = cp.random.randn(k,n)
  B = B/cp.linalg.norm(B,axis = 0)

  T = 10
  p1,o1,s1 = manopt(jnp.asarray(C),jnp.asarray(B),maxtime = T)
  print("pymanopt 1 time",s1.time,"obj",-s1.cost)

  f_rtr[rep,0] = -s1.cost
  t_rtr[rep,0] = s1.time

  T = 30
  p2,o2,s2 = manopt(jnp.asarray(C),jnp.asarray(B),maxtime = T)
  print("pymanopt 1 time",s2.time,"obj",-s2.cost)

  f_rtr[rep,1] = -s2.cost
  t_rtr[rep,1] = s2.time

  T = 60
  p3,o3,s3 = manopt(jnp.asarray(C),jnp.asarray(B),maxtime = T)
  print("pymanopt 1 time",s3.time,"obj",-s3.cost)

  f_rtr[rep,2] = -s3.cost
  t_rtr[rep,2] = s3.time


  cg_t0 = time.time()
  for t in range(400):
      B = B.dot(C)
      B = B/cp.linalg.norm(B,axis = 0)
  cp.cuda.Device().synchronize()
  cg_t1 = time.time()
  obj1 = cp.tensordot(B,B.dot(C))
  print("time",cg_t1-cg_t0)
  print("obj 1",obj1)

  f_cg[rep,0] = obj1
  t_cg[rep,0] = cg_t1-cg_t0

  cg_t2 = time.time()
  for t in range(400):
      B = B.dot(C)
      B = B/cp.linalg.norm(B,axis = 0)
  cp.cuda.Device().synchronize()
  cg_t3 = time.time()
  obj2 = cp.tensordot(B,B.dot(C))
  print("time",cg_t3-cg_t2)
  print("obj 2",obj2)

  f_cg[rep,1] = obj2
  t_cg[rep,1] = cg_t3-cg_t2 + cg_t1-cg_t0

  cg_t4 = time.time()
  for t in range(800):
      B = B.dot(C)
      B = B/cp.linalg.norm(B,axis = 0)
  cp.cuda.Device().synchronize()
  cg_t5 = time.time()
  obj3 = cp.tensordot(B,B.dot(C))
  print("time",cg_t5-cg_t4)
  print("obj 3",obj3)

  f_cg[rep,2] = obj3
  t_cg[rep,2] = cg_t5-cg_t4 + cg_t3-cg_t2 + cg_t1-cg_t0

  print("total time",cg_t5+cg_t3+cg_t1-cg_t4-cg_t2-cg_t0)
  print("diff",obj3 + s3.cost)
  print("rell diff",(obj3 + s3.cost)/obj3)

----------------REP++ 0 ++---------------------------------------
n 30000 r 245
Optimizing...
Terminated - max time reached after 8 iterations.

pymanopt 1 time 14.222194910049438 obj 530.5491822406034
n 30000 r 245
Optimizing...
Terminated - max time reached after 13 iterations.

pymanopt 1 time 47.20071601867676 obj 532.3443060714276
n 30000 r 245
Optimizing...
Terminated - max time reached after 14 iterations.

pymanopt 1 time 67.4463849067688 obj 532.3669233541971
time 10.255922794342041
obj 1 532.3532178625621
time 10.288180112838745
obj 2 532.3667024572256
time 20.677672863006592
obj 3 532.3686767002961
total time 41.221776247024536
diff 0.001753346099008013
rell diff 3.2934809573612127e-06
----------------REP++ 1 ++---------------------------------------
n 30000 r 245
Optimizing...
Terminated - max time reached after 9 iterations.

pymanopt 1 time 12.368594408035278 obj 530.8007331936753
n 30000 r 245
Optimizing...
Terminated - max time reached after 13 iterations.

pymanopt 1 t

In [ ]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu.npz",f_rtr = f_rtr,t_rtr = t_rtr,f_cg = f_cg, t_cg = t_cg)

In [ ]:
f_rtr.mean(axis = 0)

array([530.68958357, 532.32549409, 532.33835305])

In [ ]:
f_cg.mean(axis = 0)

array([532.32425496, 532.33789925, 532.33992158])

In [ ]:
t_rtr

array([[ 14.22219491,  47.20071602,  67.44638491],
       [ 12.36859441,  47.66241908,  65.22230935],
       [ 12.13752604,  45.58072567,  62.64909649],
       [ 12.3128922 ,  47.72281504,  65.2754333 ],
       [ 12.0287528 ,  46.67609715,  64.18294024],
       [ 11.95868802,  41.19925928,  92.91021776],
       [ 12.10177422,  40.04402971,  88.81331778],
       [ 12.30136347,  47.00484467,  64.59495711],
       [ 11.84149551,  46.33209944,  65.3156023 ],
       [ 12.2983737 ,  31.08735037,  65.34436154],
       [ 11.88454866,  45.43553591,  63.67000008],
       [ 12.15546727,  39.33828306,  97.6499455 ],
       [ 12.43944049,  40.52379012,  94.84468293],
       [ 12.23468161,  46.50899601,  63.13881087],
       [ 12.21322894,  40.52861452,  97.40568233],
       [ 11.88137603,  46.99589777,  66.34129429],
       [ 12.21891308,  38.94294095,  88.40017843],
       [ 12.07228756,  40.3048625 ,  96.56357932],
       [ 12.17301488,  46.13750935, 105.41493917],
       [ 11.96803284,  45.99960

In [ ]:
t_cg

array([[10.25592279, 20.54410291, 41.22177577],
       [10.25517511, 20.5567565 , 41.2587657 ],
       [10.26423097, 20.57929873, 41.30058098],
       [10.26078629, 20.56922817, 41.27803993],
       [10.25974631, 20.56888366, 41.26659799],
       [10.25299954, 20.54261637, 41.23205328],
       [10.25841594, 20.55661702, 41.25069356],
       [10.26176834, 20.57448316, 41.27828312],
       [10.26034117, 20.56413388, 41.26955485],
       [10.26534009, 20.57709908, 41.29729486],
       [10.26352358, 20.57393241, 41.2784729 ],
       [10.25419474, 20.55582929, 41.23292565],
       [10.28020358, 20.58387232, 41.27211666],
       [10.25948548, 20.56329322, 41.26457453],
       [10.25853229, 20.5568521 , 41.26238728],
       [10.26116252, 20.56703925, 41.26900864],
       [10.25914359, 20.56038284, 41.25490355],
       [10.25870824, 20.55824852, 41.2633512 ],
       [10.2591033 , 20.55841947, 41.26862097],
       [10.26510215, 20.58630824, 41.30334044],
       [10.26188135, 20.58019662, 41.292

In [ ]:
f_cg

array([[532.35321786, 532.36670246, 532.3686767 ],
       [532.4079967 , 532.42152152, 532.42353105],
       [532.50066452, 532.51431229, 532.5162148 ],
       [532.34301311, 532.35663567, 532.35880848],
       [532.41277463, 532.42586427, 532.42772131],
       [532.16459   , 532.17840204, 532.18044064],
       [532.31377521, 532.32741864, 532.32951422],
       [532.42904555, 532.44267873, 532.44467627],
       [532.27072001, 532.28485998, 532.28704294],
       [532.18963855, 532.20353517, 532.20559137],
       [532.29112758, 532.30477698, 532.30678434],
       [532.31653514, 532.33061268, 532.33267973],
       [532.17499384, 532.18893337, 532.19096931],
       [532.41075851, 532.42423143, 532.42616619],
       [532.46300234, 532.47599901, 532.47780902],
       [532.2376742 , 532.25092655, 532.25283118],
       [532.33367854, 532.34694763, 532.34892237],
       [532.37501734, 532.38858451, 532.3905961 ],
       [532.28898445, 532.30304016, 532.30506883],
       [532.4043268 , 532.41782

In [ ]:
f_rtr

array([[530.54918224, 532.34430607, 532.36692335],
       [530.80073319, 532.39715633, 532.42133712],
       [530.85807631, 532.49213895, 532.51432779],
       [530.69032261, 532.33332748, 532.35707563],
       [530.76051104, 532.4012751 , 532.42384348],
       [530.52156784, 532.17940106, 532.17940106],
       [530.72758946, 532.3285402 , 532.3285402 ],
       [530.81146449, 532.41874975, 532.44224776],
       [530.63815034, 532.26309219, 532.28492486],
       [530.61721075, 532.17830162, 532.20328558],
       [530.67002517, 532.28235605, 532.30468189],
       [530.68655258, 532.33209206, 532.33209206],
       [530.55372143, 532.190274  , 532.190274  ],
       [530.75988551, 532.40208728, 532.42399279],
       [530.78713652, 532.47715562, 532.47715562],
       [530.58355127, 532.22651889, 532.25029532],
       [530.66281958, 532.34833088, 532.34833088],
       [530.7079019 , 532.38973089, 532.38973089],
       [530.65673844, 532.28376468, 532.30379802],
       [530.79705372, 532.39362

In [ ]:
!mkdir -p /content/drive/MyDrive/result_folder
np.savez("/content/drive/MyDrive/result_folder/results_gpu_final.npz",f_rtr = f_rtr,t_rtr = t_rtr,f_cg = f_cg, t_cg = t_cg)

In [ ]:
f_cg-f_rtr

array([[ 1.80403562e+00,  2.23963858e-02,  1.75334610e-03],
       [ 1.60726350e+00,  2.43651896e-02,  2.19393195e-03],
       [ 1.64258821e+00,  2.21733476e-02,  1.88701198e-03],
       [ 1.65269050e+00,  2.33081844e-02,  1.73284860e-03],
       [ 1.65226359e+00,  2.45891696e-02,  3.87783314e-03],
       [ 1.64302216e+00, -9.99021980e-04,  1.03958155e-03],
       [ 1.58618575e+00, -1.12155634e-03,  9.74020287e-04],
       [ 1.61758106e+00,  2.39289827e-02,  2.42850991e-03],
       [ 1.63256967e+00,  2.17677875e-02,  2.11808393e-03],
       [ 1.57242781e+00,  2.52335527e-02,  2.30578377e-03],
       [ 1.62110241e+00,  2.24209284e-02,  2.10245034e-03],
       [ 1.62998257e+00, -1.47938286e-03,  5.87662149e-04],
       [ 1.62127241e+00, -1.34062593e-03,  6.95315756e-04],
       [ 1.65087300e+00,  2.21441439e-02,  2.17340744e-03],
       [ 1.67586582e+00, -1.15661584e-03,  6.53397611e-04],
       [ 1.65412293e+00,  2.44076505e-02,  2.53585835e-03],
       [ 1.67085895e+00, -1.38325030e-03